# Establecimientos Educativos — Preescolar, Básica y Media

El dataset contiene aproximadamente 588 mil de registros y un total de 24 columnas, correspondientes a los periodos entre el 2015 y el 2024.

Tener en cuenta que esta conjunto de datos contiene la información a nivel de departamento y secretaria, si requiere información a nivel municipal dirigirse al informe que se encuentra por sedes.

**Motivo de seleccion:**

Contiene la ubicación y cantidad de colegios por municipio. Con esta información se puede calcular cuántos colegios hay por cada 1.000 menores de edad, lo que ayuda a entender qué tan buena es la oferta educativa en cada zona. 

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, StringType, LongType
from pyspark.sql.window import Window

## Lectura y Descripcion del Dataset

In [0]:
# MEN_ESTABLECIMIENTOS_EDUCATIVOS_PREESCOLAR_BÁSICA_Y_MEDIA https://www.datos.gov.co/Educaci-n/MEN_ESTABLECIMIENTOS_EDUCATIVOS_PREESCOLAR_B-SICA_/cfw5-qzt5/about_data
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/workspace/pdge(saber-11)/saber11/BRONZE/establecimientos.csv")

In [0]:
df.printSchema()

In [0]:
print("Rows: ", df.count())
print("Columns: ", len(df.columns))

## Reporte de Calidad de Datos

#### Valores Nulos

In [0]:
null_count = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
print("Columnas con nulos:")
for c in null_count.columns:
    count = null_count.select(c).collect()[0][0]
    if count > 0:
        print(c, f'({count})')

#### Identificación de duplicados

In [0]:
duplicates = df.groupBy(df.columns).count().filter("count > 1")
print("Cantidad de duplicados: ", duplicates.count())

#### Tipos de dato inconsistentes

Schema

In [0]:
df.printSchema()

Analisis de datos con ejemplos

In [0]:
df.display()

Inconsistencias encontradas

Columnas que son enteros:
- "AÑO", "COD_DANE_DEPARTAMENTO", "COD_SECRETARIA", "COD_DANE_MUNICIPIO", "COD_SECTOR", "COD_CARACTER", "COD_CALENDARIO", "TOTAL_MATRICULA", "CANTIDAD_SEDES"
- CODIGO_DANE es numero, pero long


#### Propuesta de técnica para tratar los valores faltantes

| Nombre Columna     | Nulos  | Tratamiento                                                      | Justificación                                                        |
| ------------------ | ------ | ---------------------------------------------------------------- | -------------------------------------------------------------------- |
| DEPARTAMENTO       | 25     | Eliminar registros                                               | Pocos nulos (<5%), variable clave de ubicación                       |
| SECRETARIA         | 25     | Imputación por CODIGO_DANE (otros años)                          | Variable estable por institución, recuperable longitudinalmente      |
| COD_DANE_MUNICIPIO | 25     | Imputación por CODIGO_DANE + enriquecimiento por WEB             | Se puede recuperar por histórico y complementar con scraping externo |
| MUNICIPIO          | 25     | Eliminar registros                                               | Pocos nulos (<5%), redundante pero necesaria para interpretación     |
| COD_CARACTER       | 2277   | Imputación por CODIGO_DANE (otros años)                          | Variable categórica estable en el tiempo                             |
| CARACTER           | 2277   | Imputación por CODIGO_DANE (otros años)                          | Misma lógica que COD_CARACTER                                        |
| DIRECCION          | 34     | Eliminar columna                                                 | Baja relevancia analítica                                            |
| BARRIO_VEREDA      | 119585 | Eliminar columna                                                 | Alta proporción de nulos y bajo impacto                              |
| TELEFONO           | 46174  | Eliminar columna                                                 | No relevante para análisis                                           |
| FAX                | 466212 | Eliminar columna                                                 | Muy alta proporción de nulos                                         |
| EMAIL              | 34982  | Eliminar columna                                                 | No relevante para análisis                                           |
| RECTOR             | 48     | Eliminar columna                                                 | No relevante para análisis                                           |
| WEB                | 448463 | Imputación por CODIGO_DANE (otros años) + fallback "Inexistente" | Se prioriza recuperación longitudinal antes de asumir inexistencia   |
| CANTIDAD_SEDES     | 1      | Imputación por CODIGO_DANE (otros años)                          | Variable estable por institución, no requiere mediana                |


## Limpieza y transformaciones


#### Eliminacion de columnas

Se omitirán columnas que contienen datos muy especificos de los establecimientos o no relacionados con el motivo de seleccion del dataset.

In [0]:
df01 = df
# Comunicacion
DELETE_COLUMNS = ['TELEFONO', 'FAX', 'EMAIL']
# Datos especificos
DELETE_COLUMNS += ['NOMBRE_ESTABLECIMIENTO', 'DIRECCION', 'BARRIO_VEREDA', 'RECTOR']
# Datos repetidos (codigo - descripcion, se deja el mas oportuno)
# En el caso de SECRETARIA se dejan ambos (funciona mejor nombre pero se deja codigo en caso de join)
DELETE_COLUMNS += ['DEPARTAMENTO', 'MUNICIPIO', 'COD_SECTOR', 'COD_CARACTER', 'COD_CALENDARIO']
for c in DELETE_COLUMNS:
    df01 = df01.drop(c)
df01.columns


#### Correción de tipos de dato

In [0]:
df02 = df01
# Int Cast
INT_COLS = ["AÑO", "COD_DANE_DEPARTAMENTO", "COD_SECRETARIA", "COD_DANE_MUNICIPIO", "COD_SECTOR", "COD_CARACTER", "COD_CALENDARIO", "TOTAL_MATRICULA", "CANTIDAD_SEDES"]
for c in [col for col in INT_COLS if col in df02.columns]:
    df02 = df02.withColumn(
        c,
        F.regexp_replace(F.col(c).cast("string"), r"\.", "").try_cast("int")
    )
# Long Cast
LONG_COLS = ["CODIGO_DANE"]
for c in [col for col in LONG_COLS if col in df02.columns]:
    df02 = df02.withColumn(c, F.regexp_replace(F.col(c), r"\.", "").try_cast(LongType()))

#### Tratamiento de nulos

Primero se rectifica el estado de nulos luego de los tratamientos anteriores

In [0]:
null_count = df02.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df02.columns])
print("Columnas con nulos:")
for c in null_count.columns:
    count = null_count.select(c).collect()[0][0]
    if count > 0:
        print(c, f'({count})')

Se realizó una imputación de valores nulos aprovechando la naturaleza longitudinal del dataset, donde cada institución (CODIGO_DANE) aparece en múltiples años. Para las variables consideradas estables en el tiempo (SECRETARIA, COD_DANE_MUNICIPIO, CARACTER, CANTIDAD_SEDES), se aplicó una estrategia bidireccional usando funciones de ventana en PySpark: primero se intentó completar los nulos con el último valor no nulo de años anteriores (forward fill) y, en caso de no existir, con el primer valor disponible en años posteriores (backward fill), priorizando siempre el valor original.

In [0]:
df03 = df02
# columnas a imputar
COLS_TO_IMPUTE = [
    "SECRETARIA",
    "COD_DANE_MUNICIPIO",
    "CARACTER",
    "CANTIDAD_SEDES"
]

# ventana por colegio
w_forward = Window.partitionBy("CODIGO_DANE").orderBy("AÑO").rowsBetween(Window.unboundedPreceding, 0)
w_backward = Window.partitionBy("CODIGO_DANE").orderBy("AÑO").rowsBetween(0, Window.unboundedFollowing)

for c in COLS_TO_IMPUTE:
    df03 = df03.withColumn(
        c,
        F.coalesce(
            F.col(c),
            F.last(F.col(c), ignorenulls=True).over(w_forward),
            F.first(F.col(c), ignorenulls=True).over(w_backward)
        )
    )

In [0]:
null_count = df01.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df01.columns])
print("Columnas con nulos:")
for c in null_count.columns:
    count = null_count.select(c).collect()[0][0]
    if count > 0:
        print(c, f'({count})')

Nulos SECRETARIA

Todos los nulos de secretaria tienen COD_SECRETARIA 7738 que corresponden a la Resolución 7738 del 22 de junio de 2018 del ICBF (Instituto Colombiano de Bienestar Familiar). Por lo que se completan.

In [0]:
df04 = df03
df04 = df04.withColumn(
    "SECRETARIA",
    F.when(F.col("SECRETARIA").isNull(), "BIENESTAR FAMILIAR").otherwise(F.col("SECRETARIA"))
)

Nulos COD_DANE_MUNICIPIO 

Dado que son pocas columnas y el valor del municipio es importante para el objetivo, se busco en las paginas de los registros que aparecen y se determina que todos pertenecen al municipio de la estrella con codigo 5380. Los que tienen web null se imputan con ese municipio tambien porque coinciden en departamento. Podria decirse que es imputacion por moda.

In [0]:
df05 = df04
df05 = df05.withColumn("COD_DANE_MUNICIPIO", F.when(F.col("COD_DANE_MUNICIPIO").isNull(), 5380).otherwise(F.col("COD_DANE_MUNICIPIO")))

#### Tratamiento de columnas

CARACTER

Caracter ademas de tener nulos, hay valores "-" por lo que al no se un dato fundamental de negocio, se imputara con "Desconocido"

In [0]:
df06 = df05
df06 = df06.withColumn("CARACTER", F.when((F.col("CARACTER").isNull()) | (F.col("CARACTER") == "-"), "DESCONOCIDO").otherwise(F.upper(F.col("CARACTER"))))

WEB

Se asume que si el valor de esta columna no empieza por www sea porque esta nulo o porque se escribió directamente no tiene o algun otro valor, no tiene.


In [0]:
df07 = df06
df07 = df07.withColumn("TIENE_WEB", F.when((F.col("WEB").isNull()) | (~F.col("WEB").startswith("www")), False).otherwise(True)).drop("WEB")

In [0]:
df_final = df07
print("Rows: ", df_final.count())
print("Columns: ", len(df_final.columns))
print("Rows with null: ",df_final.filter(sum(F.col(c).isNull().cast("int") for c in df_final.columns) > 0).count())
df_final.display()

In [0]:
df_final = df_final.withColumnRenamed("COD_DANE_MUNICIPIO","COD_MUNICIPIO" )

In [0]:
df_final.write.mode("overwrite").parquet("/Volumes/workspace/pdge(saber-11)/saber11/SILVER/establecimientos/")

In [0]:
# Descriptiva de dataset
# =========================================
# DESCRIPTIVA TERRITORIAL POR DEPARTAMENTO
# =========================================

from pyspark.sql import functions as F
from pyspark.sql.types import LongType
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

df00 = df

# -----------------------------------------
# 0. CASTEO (SE MANTIENE TAL CUAL)
# -----------------------------------------
INT_COLS = [
    "AÑO", "COD_DANE_DEPARTAMENTO", "COD_SECRETARIA", "COD_DANE_MUNICIPIO",
    "COD_SECTOR", "COD_CARACTER", "COD_CALENDARIO", "TOTAL_MATRICULA",
    "CANTIDAD_SEDES"
]

for c in [col for col in INT_COLS if col in df00.columns]:
    df00 = df00.withColumn(
        c,
        F.regexp_replace(F.col(c).cast("string"), r"\.", "").try_cast("int")
    )

LONG_COLS = ["CODIGO_DANE"]
for c in [col for col in LONG_COLS if col in df00.columns]:
    df00 = df00.withColumn(
        c,
        F.regexp_replace(F.col(c), r"\.", "").try_cast(LongType())
    )

# -----------------------------------------
# 1. SELECCIÓN
# -----------------------------------------
cols_needed = [
    "AÑO",
    "CODIGO_DANE",
    "NOMBRE_ESTABLECIMIENTO",
    "COD_SECTOR",
    "SECTOR",
    "COD_CARACTER",
    "CARACTER",
    "TOTAL_MATRICULA",
    "CANTIDAD_SEDES",
    "DEPARTAMENTO",
    "COD_DANE_DEPARTAMENTO"
]

df = df00.select(*[c for c in cols_needed if c in df00.columns])

# -----------------------------------------
# 2. FILTRO
# -----------------------------------------
df = df.filter((F.col("AÑO") >= 2015) & (F.col("AÑO") <= 2023))

# -----------------------------------------
# 3. LIMPIEZA
# -----------------------------------------
df = (
    df
    .withColumn("DEPARTAMENTO", F.trim(F.col("DEPARTAMENTO")))
    .withColumn("NOMBRE_ESTABLECIMIENTO", F.trim(F.col("NOMBRE_ESTABLECIMIENTO")))
    .withColumn("SECTOR_STD", F.upper(F.trim(F.col("SECTOR"))))
    .withColumn("CARACTER_STD", F.upper(F.trim(F.col("CARACTER"))))
)

# -----------------------------------------
# 4. INDICADORES
# -----------------------------------------
df = (
    df
    .withColumn(
        "ES_OFICIAL_ROW",
        F.when((F.col("COD_SECTOR") == 1001) | (F.col("SECTOR_STD") == "OFICIAL"), 1).otherwise(0)
    )
    .withColumn(
        "ES_NO_OFICIAL_ROW",
        F.when((F.col("COD_SECTOR") == 1002) | (F.col("SECTOR_STD") == "NO OFICIAL"), 1).otherwise(0)
    )
    .withColumn(
        "ES_TECNICO_ROW",
        F.when(
            (F.col("COD_CARACTER") == 818) |
            (F.col("COD_CARACTER") == 2432) |
            (F.col("CARACTER_STD").contains("TÉCNICO")) |
            (F.col("CARACTER_STD").contains("TECNICO")),
            1
        ).otherwise(0)
    )
)

# -----------------------------------------
# 5. CLAVE INSTITUCIÓN-AÑO
# -----------------------------------------
inst_keys = ["AÑO", "CODIGO_DANE", "NOMBRE_ESTABLECIMIENTO"]

# -----------------------------------------
# 6. DEPARTAMENTO CANÓNICO
# -----------------------------------------
location_counts = (
    df.groupBy(*inst_keys, "COD_DANE_DEPARTAMENTO", "DEPARTAMENTO")
    .agg(F.count("*").alias("n_rows"))
)

w = Window.partitionBy(*inst_keys).orderBy(
    F.desc("n_rows"),
    F.asc("COD_DANE_DEPARTAMENTO"),
    F.asc("DEPARTAMENTO")
)

canonical_dept = (
    location_counts
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "n_rows")
)

# -----------------------------------------
# 7. MÉTRICAS POR INSTITUCIÓN-AÑO
# -----------------------------------------
institution_metrics = (
    df.groupBy(*inst_keys)
    .agg(
        F.max("TOTAL_MATRICULA").alias("total_matricula_inst"),
        F.max("CANTIDAD_SEDES").alias("cantidad_sedes_inst"),
        F.max("ES_OFICIAL_ROW").alias("es_oficial"),
        F.max("ES_NO_OFICIAL_ROW").alias("es_no_oficial"),
        F.max("ES_TECNICO_ROW").alias("es_tecnico")
    )
)

# -----------------------------------------
# 8. TABLA FINAL INSTITUCIÓN-AÑO
# -----------------------------------------
institution_year = canonical_dept.join(
    institution_metrics,
    on=inst_keys,
    how="inner"
)

# -----------------------------------------
# 9. AGREGACIÓN DEPARTAMENTO-AÑO
# -----------------------------------------
dept_year = (
    institution_year
    .groupBy("AÑO", "COD_DANE_DEPARTAMENTO", "DEPARTAMENTO")
    .agg(
        F.count("*").alias("n_establecimientos"),
        F.sum("total_matricula_inst").alias("matricula_total"),
        F.sum("cantidad_sedes_inst").alias("sedes_totales"),
        F.avg("total_matricula_inst").alias("matricula_promedio_establecimiento"),
        F.avg("cantidad_sedes_inst").alias("sedes_promedio_establecimiento"),
        F.sum("es_oficial").alias("n_oficiales"),
        F.sum("es_no_oficial").alias("n_no_oficiales"),
        F.sum("es_tecnico").alias("n_tecnicos")
    )
    .withColumn(
        "pct_oficial",
        F.when(F.col("n_establecimientos") > 0, F.col("n_oficiales") / F.col("n_establecimientos"))
    )
    .withColumn(
        "pct_no_oficial",
        F.when(F.col("n_establecimientos") > 0, F.col("n_no_oficiales") / F.col("n_establecimientos"))
    )
    .withColumn(
        "pct_tecnico",
        F.when(F.col("n_establecimientos") > 0, F.col("n_tecnicos") / F.col("n_establecimientos"))
    )
    .withColumn(
        "matricula_por_sede",
        F.when(F.col("sedes_totales") > 0, F.col("matricula_total") / F.col("sedes_totales"))
    )
)

# -----------------------------------------
# 10. VALIDACIONES GENERALES
# -----------------------------------------
print("Filas crudas:", df.count())
print("Filas institución-año:", institution_year.count())
print("Filas departamento-año:", dept_year.count())

print("\nSchema dept_year:")
dept_year.printSchema()

# -----------------------------------------
# 11. TABLA DESCRIPTIVA GENERAL NUMÉRICA
# -----------------------------------------
print("\n=== Descriptiva general de variables numéricas ===")
dept_year.select(
    "n_establecimientos",
    "matricula_total",
    "sedes_totales",
    "matricula_promedio_establecimiento",
    "sedes_promedio_establecimiento",
    "pct_oficial",
    "pct_no_oficial",
    "pct_tecnico",
    "matricula_por_sede"
).describe().show(truncate=False)

# -----------------------------------------
# 12. TABLA 1: TOP DEPARTAMENTOS POR MATRÍCULA EN 2023
# -----------------------------------------
print("\n=== Top 15 departamentos por matrícula total en 2023 ===")
top_matricula_2023 = (
    dept_year
    .filter(F.col("AÑO") == 2023)
    .orderBy(F.desc("matricula_total"))
    .select(
        "DEPARTAMENTO",
        "n_establecimientos",
        "matricula_total",
        "sedes_totales",
        "pct_oficial",
        "pct_tecnico",
        "matricula_por_sede"
    )
)

top_matricula_2023.show(15, truncate=False)

# -----------------------------------------
# 13. TABLA 2: TOP DEPARTAMENTOS POR % OFICIAL EN 2023
# -----------------------------------------
print("\n=== Top 15 departamentos por proporción de establecimientos oficiales en 2023 ===")
top_oficial_2023 = (
    dept_year
    .filter((F.col("AÑO") == 2023) & (F.col("n_establecimientos") >= 10))
    .orderBy(F.desc("pct_oficial"), F.desc("matricula_total"))
    .select(
        "DEPARTAMENTO",
        "n_establecimientos",
        "matricula_total",
        "pct_oficial",
        "pct_no_oficial",
        "pct_tecnico"
    )
)

top_oficial_2023.show(15, truncate=False)

# -----------------------------------------
# 14. TABLA 3: EVOLUCIÓN ANUAL NACIONAL
# -----------------------------------------
print("\n=== Evolución anual nacional de la oferta educativa (2015-2023) ===")
national_year = (
    dept_year
    .groupBy("AÑO")
    .agg(
        F.sum("n_establecimientos").alias("n_establecimientos_total"),
        F.sum("matricula_total").alias("matricula_total_nacional"),
        F.sum("sedes_totales").alias("sedes_totales_nacional"),
        F.avg("pct_oficial").alias("pct_oficial_promedio_departamental"),
        F.avg("pct_tecnico").alias("pct_tecnico_promedio_departamental")
    )
    .orderBy("AÑO")
)

national_year.show(truncate=False)

# -----------------------------------------
# 15. TABLA 4: DEPARTAMENTOS CON MAYOR MATRÍCULA POR SEDE EN 2023
# -----------------------------------------
print("\n=== Top 15 departamentos por matrícula por sede en 2023 ===")
top_presion_2023 = (
    dept_year
    .filter((F.col("AÑO") == 2023) & (F.col("sedes_totales") > 0))
    .orderBy(F.desc("matricula_por_sede"))
    .select(
        "DEPARTAMENTO",
        "n_establecimientos",
        "matricula_total",
        "sedes_totales",
        "matricula_por_sede",
        "pct_oficial"
    )
)

top_presion_2023.show(15, truncate=False)

# -----------------------------------------
# 16. GRÁFICA: TOP DEPARTAMENTOS POR MATRÍCULA 2023
# -----------------------------------------
pdf_top_dept = (
    dept_year
    .filter(F.col("AÑO") == 2023)
    .select(
        "DEPARTAMENTO",
        "matricula_total",
        "n_establecimientos",
        "pct_oficial",
        "pct_tecnico"
    )
    .orderBy(F.desc("matricula_total"))
    .limit(15)
    .toPandas()
)

pdf_top_dept = pdf_top_dept.sort_values("matricula_total", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(pdf_top_dept["DEPARTAMENTO"], pdf_top_dept["matricula_total"])
plt.xlabel("Matrícula total")
plt.ylabel("Departamento")
plt.title("Top 15 departamentos por matrícula total (2023)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()